# NB6 — locked test, decision and report

Scores every registered model exactly once on every locked test file (both GPUs). When everything is scored, applies the gates to the primary candidate (accept, or keep the Stage-A model) and writes the report.

**Accelerator:** GPU T4 x2 · **Internet:** OFF · **Attach:** code, model, NB0, NB2, NB5, all NB3 and NB4 outputs (+ any earlier NB6 output)

Run with **Save Version → Save & Run All (Commit)**. Afterwards: if it says "Not finished", run it again with this output attached too. Otherwise read `report/report.md`.

In [ ]:
SESSION_HOURS = 12   # your GPU session limit (see the notebook settings panel)
# --- Setup (identical in every notebook; nothing to edit) --------------------
import json, os, shutil, subprocess, sys, time
from pathlib import Path
NOTEBOOK_STARTED = time.time()
KAGGLE = Path(os.environ.get('PCL_KAGGLE_ROOT', '/kaggle'))   # override only for a local rehearsal
INPUT, WORKING = KAGGLE / 'input', KAGGLE / 'working'

code_roots = sorted({p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'pcl' / 'cli.py').is_file()})
assert len(code_roots) == 1, f'attach exactly one copy of the pcl code dataset; found {code_roots}'
WORK = Path(os.environ.get('PCL_WORK', '/tmp/protected-cl'))
if not WORK.exists():
    shutil.copytree(code_roots[0], WORK, ignore=shutil.ignore_patterns('outputs', '__pycache__', '*.egg-info', '.pytest_cache'))
sys.path.insert(0, str(WORK))   # `python -m pcl` runs from WORK, so no package install is needed

import importlib.metadata as metadata
def _version(name):
    try:
        return tuple(int(x) for x in metadata.version(name).split('.')[:2])
    except Exception:
        return (0,)
needed = {'transformers': (4, 51), 'safetensors': (0, 4), 'accelerate': (1, 0), 'PyYAML': (6, 0)}
missing = [f'{n}>={".".join(map(str, v))}' for n, v in needed.items() if _version(n) < v]
if missing:
    print('Installing', missing, '(needs Internet ON for this notebook)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import yaml
from pcl import kaggle
from pcl.io import read_json

def pcl(*args):
    """Run one pcl command in a fresh process (its GPU memory is released when it exits)."""
    print('$ pcl', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pcl', *map(str, args)], check=True, cwd=WORK)

def on_gpus(commands, log_dir):
    """One pcl command per T4, logs streamed live; fails if either command fails."""
    commands = [json.dumps([sys.executable, '-m', 'pcl', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

def show(path):
    try:
        from IPython.display import Markdown, display
        display(Markdown(Path(path).read_text()))
    except ImportError:
        print(Path(path).read_text())

def remaining_seconds():
    """Session time left, from SESSION_HOURS in the first cell."""
    return int(SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_STARTED))

found = kaggle.discover(INPUT, exclude=code_roots[0])
print({key: len(value) for key, value in found.items()})

In [ ]:
MODEL = kaggle.one(found, 'models', 'pinned model Dataset (made from the NB1 output)')
M0_CONFIG = kaggle.m0_config(MODEL, WORKING / 'configs' / 'm0.yaml')
TESTS = kaggle.one(found, 'tests', 'NB5 output (tests)')
REGISTRY = WORKING / 'registry'
if not REGISTRY.exists():
    shutil.copytree(kaggle.latest_registry(found), REGISTRY)
study = read_json(REGISTRY / 'study.json')
batch = yaml.safe_load((kaggle.one(found, 'pilot', 'NB2 output') / 'study.yaml').read_text())['eval_batch_size']
artifacts = sorted(study['artifacts'])
commands = [['evaluate-locked', '--registry', REGISTRY, '--artifact-id', *half, '--model-config', M0_CONFIG,
             '--test-root', TESTS, '--search', INPUT, '--batch-size', batch, '--session-seconds', remaining_seconds()]
            for half in (artifacts[0::2], artifacts[1::2]) if half]
on_gpus(commands, WORKING / 'logs')
scored = sum(read_json(p)['status'] == 'completed' for p in (REGISTRY / 'locked').glob('*.json'))
total = len(artifacts) * len(study['test_files'])
print(scored, 'of', total, 'scored')

In [ ]:
if scored < total:
    print('Not finished: run NB6 again with this output attached as an extra input.')
else:
    if not list(REGISTRY.glob('decision-*.json')):
        pcl('decide', '--registry', REGISTRY, '--smoke-report', kaggle.one(found, 'smoke', 'NB0 output'), '--search', INPUT)
    pcl('report', '--registry', REGISTRY, '--out', WORKING / 'report')
    show(WORKING / 'report' / 'report.md')